# Universal Multi-Sequence, Multi-Center and Multi-View CMR Segmentation Challenge

## Final Model Report and Submission Package

This notebook summarizes the final Task 1 + Task 2 workflow, the best submissions tried so far, the selected Model A / Model B / Main Model story, training curves, local validation metrics, and qualitative inference overlays.

**Fresh final upload file:**

`/Users/salmaheshamsalem/Desktop/DEEP_LEARNING_PROJECT/SUBMISSIONS_ORGANIZED/UPLOAD_THIS/FINAL_MAIN_TASK1_TASK2.zip`

**Best public score observed so far:** `0.64`.


## 1. What the Competition Scores

The challenge evaluates two tasks:

### Task 1: Cine MRI
- Segmentation Dice similarity coefficient.
- Hausdorff distance.
- Average surface distance.
- Ejection fraction Pearson correlation.

### Task 2: LGE MRI
- Segmentation Dice similarity coefficient.
- Scar mass Pearson correlation.
- Scar mass relative absolute error.

Correct upload structure is critical. A valid zip must contain:

```text
submission.zip
├── task1_cine/
│   ├── SAX/*.nii.gz
│   ├── 2CH/*.nii.gz
│   ├── 4CH/*.nii.gz
│   └── ef_predictions.json
└── task2_lge/
    ├── SAX/*.nii.gz
    ├── 2CH/*.nii.gz
    ├── 4CH/*.nii.gz
    ├── RAS/*.nii.gz
    └── mass_predictions.json
```

Earlier failures came mostly from filename/shape mismatches. The final workflow explicitly validates file counts, labels, and spatial shapes before upload.


## 2. Step-by-Step Final Workflow

1. **Audited all previous packages and public scores.** The best public score so far is `0.64` from the Task 1 local-selected package combined with the saved Task 2 LGE model.
2. **Stopped long-running fine-tuning.** The later Task 1 epochs improved training loss but did not improve validation Dice.
3. **Rebuilt Task 1 masks.** The Task 1 main package selects the best local candidate per case/label from the strongest trained candidates.
4. **Re-ran Task 2 inference.** The saved LGE checkpoint was applied to all Task 2 validation views: `2CH`, `4CH`, `SAX`, and `RAS`.
5. **Created a fresh final zip.** The new file is `FINAL_MAIN_TASK1_TASK2.zip`.
6. **Validated the final zip.** Counts, JSON files, labels, and top-level structure were checked.
7. **Generated this notebook.** It contains training logs, score charts, local Dice charts, model descriptions, and inference overlays.


## 3. Public Score Progression

The plot below shows the public CodaBench score progression from early working submissions to the current best group.


![Submission Score Progression](notebook_assets/codabench_score_progression.png)


In [ ]:
import pandas as pd
scores = pd.read_csv('data/codabench_scores.csv')
scores.sort_values('overall_score', ascending=False)


## 4. Selected Models for Presentation

We should present three clear systems:

### Model A — Multi-View Attention U-Net
- **Purpose:** strong 2D attention-based candidate used in the final Task 1 selection pool.
- **Architecture:** ResNet-style encoder, CBAM attention, attention-gated decoder, multi-view output handling.
- **Strength:** contributed several high-quality local cases/classes, especially some long-axis structures.

### Model B — Cine Multi-View V2
- **Purpose:** cine-specialized model with stable EF JSON and useful Task 1 masks.
- **Architecture:** residual downsampling encoder, ASPP bottleneck, attention upsampling decoder, separate outputs for cine views.
- **Strength:** gave a strong early public result and supplied useful EF predictions.

### Main Model — Local-Selected Task 1 + LGE Task 2
- **Purpose:** final competition package.
- **Architecture:** model-selection ensemble for Task 1 plus a multi-head ResUNet++-style LGE model for Task 2.
- **Strength:** best public score group so far: `0.64`.

The Main Model is not just one checkpoint. It is a pragmatic competition system: use each trained candidate where it is locally strongest, then combine it with the best saved Task 2 LGE model.


## 5. Main Pipeline Architecture

The final pipeline uses shape-safe preprocessing and view-specific heads to avoid the earlier scoring crashes.


![Main Inference Pipeline](notebook_assets/architecture_pipeline.png)


## 6. Detailed Architecture Notes

### Task 1 Main Selection System
- Candidate prediction packages were produced by several trained Task 1 models.
- For each validation case and label, the candidate with the best local Dice was selected.
- This gives a higher local macro Dice than any single checkpoint.
- Ejection fraction predictions are copied from the strongest stable EF source.

### Task 2 LGE Model
- Core model: `MultiHeadResUNetPP2D`.
- Input: 2.5D slices using previous/current/next frames as channels.
- Encoder: residual convolution blocks with Squeeze-and-Excitation recalibration.
- Bottleneck: ASPP for multi-scale context.
- Decoder: attention-gated skip connections.
- Heads: separate source heads for `2CH`, `4CH`, `SAX`, and `RAS` because label spaces differ by view.
- Loss: one-vs-rest Dice/BCE/focal-style optimization for foreground-heavy segmentation.


## 7. Local Validation Metrics

The local metrics explain why the final package is the strongest current option.


![Local Dice by View](notebook_assets/local_dice_by_view.png)


In [ ]:
import json
summary = json.load(open('data/final_summary.json'))
summary


## 8. Training Curves

The Task 1 fine-tune plateaued. Epoch 2 reached `0.8616` validation MeanDice; epoch 3 dropped slightly to `0.8612`. That is why we stopped the long run and avoided spending more time on the same model family.


![Training Curves](notebook_assets/training_curves.png)


## 9. Main Model Candidate Contributions

The final Task 1 selection combines strengths across candidate families instead of forcing one checkpoint to solve every view and label.


![Candidate Selection Counts](notebook_assets/candidate_selection_counts.png)


## 10. Qualitative Inference Visuals

Each row below shows real local validation images and final predictions from `FINAL_MAIN_TASK1_TASK2.zip`.

Color overlays indicate segmentation labels. The panels compare image + ground truth, image + final prediction, and prediction-only mask.


### Task 1 Cine SAX

![Task 1 Cine SAX](notebook_assets/inference_task1_sax_overlay.png)


### Task 1 Cine 4CH

![Task 1 Cine 4CH](notebook_assets/inference_task1_4ch_overlay.png)


### Task 2 LGE SAX

![Task 2 LGE SAX](notebook_assets/inference_task2_sax_overlay.png)


### Task 2 LGE RAS

![Task 2 LGE RAS](notebook_assets/inference_task2_ras_overlay.png)


## 11. Final Submission Validation

Fresh final package:

`/Users/salmaheshamsalem/Desktop/DEEP_LEARNING_PROJECT/SUBMISSIONS_ORGANIZED/UPLOAD_THIS/FINAL_MAIN_TASK1_TASK2.zip`

Validation checks passed:

- Top-level folders: `metadata`, `task1_cine`, `task2_lge`.
- Task 1 files: `45/45` NIfTI masks.
- Task 2 files: `35/35` NIfTI masks.
- `ef_predictions.json` present with 15 keys.
- `mass_predictions.json` present with 7 keys.
- No `__MACOSX` or `.DS_Store` junk files.
- Task 1 local macro Dice: `0.8975`.
- Task 2 local mean Dice: `0.6639`.


## 12. Recommended Presentation Narrative

1. **Problem:** multi-view CMR segmentation has different label spaces and shape requirements per modality/view.
2. **Early issue:** invalid zip mappings caused shape mismatch crashes; fixing this was mandatory before modeling mattered.
3. **Model A:** attention U-Net candidate contributed strong long-axis segmentations.
4. **Model B:** cine-specific multi-view model improved robustness and EF behavior.
5. **Main Model:** final system combines the best Task 1 predictions per local case/label and pairs them with the best saved LGE Task 2 model.
6. **Outcome:** public score improved from early `0.30–0.43` range to the current best group at `0.64`.
7. **Engineering control:** every final zip is shape-checked and count-checked before upload.
